In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get('RIDESHARE_RAW_DIR', str(PROJECT_DIR / 'data/raw/fhvhv')))
PROCESSED_DIR = Path(os.environ.get('RIDESHARE_PROCESSED_DIR', str(PROJECT_DIR / 'data/processed')))
ZONE_DIR = Path(os.environ.get('RIDESHARE_ZONE_DIR', str(PROJECT_DIR / 'data/reference/taxi_zones')))
WEATHER_DIR = Path(os.environ.get('RIDESHARE_WEATHER_DIR', str(PROJECT_DIR / 'data/reference/weather_hourly')))
ANALYSIS_DIR = PROJECT_DIR / 'outputs/demand_analysis'
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.

# All directories are configured here. Hide this cell to hide local paths.
from pathlib import Path
DATA_DIR = PROCESSED_DIR
REPAIR_DIR = DATA_DIR / 'repair_2024_2025'
ZONE_DIR = ZONE_DIR
RESULTS_DIR = ANALYSIS_DIR
YEARS = [2019, 2021, 2022, 2023, 2024, 2025]
# Read validated repair candidates in place; never promote or overwrite originals.
USE_VALIDATED_REPAIRS = True
SAVE_FIGURES = True


# Spatial patterns in pickup and drop-off totals

## Scope and interpretation

These notebooks analyze **cleaned trip totals**, not trip duration, raw requests,
vehicle availability, or a causal policy effect. Combined tables do not identify platforms.
2019 includes all recorded HVFHV platforms; do not label the series Uber-only.
January 2019 and all of 2020 are excluded. Annual figures describe available months;
use common-month comparisons for cross-year conclusions.

Pickup tables use pickup calendar date/hour. Drop-off tables for 2019/2021–2023
use actual drop-off calendar date/hour. The 2024–2025 drop-off tables instead use
pickup date with drop-off hour. Consequently, they describe destination activity
for a pickup-date cohort. February 2019 and January 2021 drop-offs have missing
prior-month spill-in and are boundary-limited despite full calendar-date coverage.
Hour labels are local clock labels; repeated daylight-saving hours are combined.
Absent zone-hours are treated as zero only within included, complete months.

Validated staged drop-off repairs may be read in place using the configuration
flag. This does not replace any original analytical output. All directories are
defined in the first code cell. Dependencies: pandas, numpy, matplotlib, duckdb;
the spatial notebook also needs geopandas and its shapefile-reading dependencies.


In [ ]:
import calendar
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
db = duckdb.connect()
db.execute("SET threads=1")
db.execute("SET memory_limit='256MB'")
db.execute("SET preserve_insertion_order=false")
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 110})
def finish(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(RESULTS_DIR / (name + '.png'), dpi=180, bbox_inches='tight')
    plt.show()


In [ ]:
NOTEBOOK_ID = '05_spatial'
DIRECTIONS = ['pickup', 'dropoff']


In [ ]:
# Validate directory selection and key/date/count integrity before analysis.
stems = {'pickup': 'fhvhv_by_zone_hour', 'dropoff': 'fhvhv_by_DOzone_hour'}
repair_tasks = {('dropoff', 2025, 8), ('dropoff', 2025, 10)}
manifest, daily_frames, zone_frames, profile_frames = [], [], [], []
for year in YEARS:
    for month in range(2 if year == 2019 else 1, 13):
        for direction, stem in stems.items():
            if direction not in DIRECTIONS:
                continue
            item = dict(direction=direction, year=year, month=month)
            folder = DATA_DIR / f'{stem}_{year}_{month:02d}.parquet'
            if USE_VALIDATED_REPAIRS and (direction, year, month) in repair_tasks:
                candidate = REPAIR_DIR / folder.name
                marker = candidate / '_repair_validation.json'
                if marker.exists():
                    record = json.loads(marker.read_text(encoding='utf-8'))
                    contract = record['contract']
                    assert record['passed'] and contract['version'] == 'selected-repairs-v1-legacy-dropoff-date'
                    assert (contract['family'], contract['year'], contract['month']) == (direction, year, month)
                    folder = candidate
            if (direction, year, month) in repair_tasks and folder.parent == DATA_DIR:
                item.update(status='excluded_unrepaired', path=str(folder))
                manifest.append(item)
                continue
            parts = sorted(folder.glob('*.parquet'))
            if not parts or not (folder / '_SUCCESS').exists():
                item.update(status='missing_or_incomplete', path=str(folder))
                manifest.append(item)
                continue
            if year in [2019, 2021, 2022, 2023]:
                contract = json.loads((folder / '_aggregation_contract.json').read_text(encoding='utf-8'))
                assert contract['version'] == 'earlier-years-hourly-v2'
                assert (contract['family'], contract['year'], contract['month']) == (direction, year, month)
            zone = 'PULocationID' if direction == 'pickup' else 'DOLocationID'
            hour = 'hour' if direction == 'pickup' else 'do_hour'
            db.read_parquet([str(p) for p in parts]).create_view('source', replace=True)
            bad = db.execute(f"""SELECT count(*) FROM source WHERE
                "{zone}" IS NULL OR date IS NULL OR "{hour}" IS NULL
                OR "{hour}" NOT BETWEEN 0 AND 23 OR trip_count IS NULL OR trip_count<=0
                OR year(date)!={year} OR month(date)!={month}""").fetchone()[0]
            duplicates = db.execute(f"""SELECT count(*) FROM
                (SELECT 1 FROM source GROUP BY "{zone}",date,"{hour}" HAVING count(*)>1)""").fetchone()[0]
            dates = db.execute('SELECT count(DISTINCT date) FROM source').fetchone()[0]
            assert bad == duplicates == 0, f'Invalid rows/duplicate keys: {folder.name}'
            assert dates == calendar.monthrange(year, month)[1], f'Incomplete dates: {folder.name}'
            time_basis = ('pickup_date' if direction == 'pickup' else
                          'actual_dropoff_date' if year < 2024 else 'pickup_date_with_dropoff_hour')
            item.update(status='included', path=str(folder), time_basis=time_basis,
                        boundary_limited=direction=='dropoff' and (year,month) in [(2019,2),(2021,1)])
            manifest.append(item)
            for frames, sql in [
                (daily_frames, 'SELECT date, sum(trip_count) trips FROM source GROUP BY date'),
                (zone_frames, f'SELECT "{zone}" zone_id, sum(trip_count) trips FROM source GROUP BY "{zone}"'),
                (profile_frames, f'SELECT dayofweek(date) AS dow, "{hour}" AS "hour", sum(trip_count) AS trips FROM source GROUP BY dayofweek(date),"{hour}"')]:
                frame = db.execute(sql).df()
                for key, value in [('direction',direction),('year',year),('month',month),('time_basis',time_basis)]:
                    frame[key] = value
                frames.append(frame)

manifest = pd.DataFrame(manifest)
display(manifest.groupby(['direction','year','status']).size().rename('months').to_frame())
display(manifest[manifest.status != 'included'])
assert daily_frames, 'No complete inputs selected'
daily = pd.concat(daily_frames, ignore_index=True)
daily['date'] = pd.to_datetime(daily.date)
zone_month = pd.concat(zone_frames, ignore_index=True)
profiles = pd.concat(profile_frames, ignore_index=True)
manifest.to_csv(RESULTS_DIR / f'{NOTEBOOK_ID}_input_manifest.csv', index=False)
daily.to_csv(RESULTS_DIR / f'{NOTEBOOK_ID}_daily_totals.csv', index=False)
print('Only included months enter plots. Missing months are not zero demand.')


## Load the taxi-zone map
The join uses LocationID. Unmatched/unknown zones remain in total-demand tables, but cannot be mapped. Map values are trips/day, not trips per unit area.

In [ ]:
import geopandas as gpd
shape_files=sorted(ZONE_DIR.glob('*.shp'))
assert len(shape_files)==1, 'Expected one zone shapefile in ZONE_DIR'
zones=gpd.read_file(shape_files[0])
assert 'LocationID' in zones.columns, 'Shapefile must contain LocationID'
zones['LocationID'] = pd.to_numeric(zones['LocationID'], errors='raise')
assert zones['LocationID'].notna().all(), 'Missing zone identifiers in shapefile'
# Some taxi zones have separate polygon records. Union their geometry by ID.
attributes = [c for c in ['zone', 'borough'] if c in zones.columns]
for column in attributes:
    assert zones.groupby('LocationID')[column].nunique(dropna=False).le(1).all(), f'Conflicting {column} labels within a zone'
polygon_records = len(zones)
zones = zones[['LocationID', *attributes, 'geometry']].dissolve(by='LocationID', as_index=False, aggfunc='first')
assert zones['LocationID'].is_unique
print(f'Merged {polygon_records} polygon records into {len(zones)} distinct mapped zones.')
display(zones.drop(columns='geometry').head())
included=manifest[manifest.status=='included']
common_months=sorted(set.intersection(*included.groupby(['direction','year']).month.apply(set).tolist()))
print('Using common months for spatial year comparisons:',common_months)
z=zone_month[zone_month.month.isin(common_months)].groupby(['direction','year','zone_id'],as_index=False).trips.sum()
days=daily[daily.month.isin(common_months)].groupby(['direction','year']).date.nunique().rename('days').reset_index()
z=z.merge(days,on=['direction','year'])
z['trips_per_day']=z.trips/z.days
z['share']=z.trips/z.groupby(['direction','year']).trips.transform('sum')
unmapped=z[~z.zone_id.isin(zones.LocationID)]
display(unmapped.groupby(['direction','year']).trips.sum().rename('unmapped_trips').to_frame())
z.to_csv(RESULTS_DIR/'05_zone_demand.csv',index=False)


In [ ]:
# Shared color scale across years within each direction.
years=sorted(z.year.unique())
for direction in DIRECTIONS:
    subset=z[z.direction==direction]
    vmax=subset.trips_per_day.max()
    fig,axes=plt.subplots(1,len(years),figsize=(5*len(years),7),squeeze=False)
    for ax,year in zip(axes.flat,years):
        layer=zones.merge(subset[subset.year==year],left_on='LocationID',right_on='zone_id',how='left')
        layer.plot(column='trips_per_day',ax=ax,cmap='Blues',vmin=0,vmax=vmax,legend=True,
                   missing_kwds={'color':'lightgrey','label':'No observed row'},edgecolor='#808080',linewidth=.2)
        ax.set_title(f'{direction.title()} {year}');ax.set_axis_off()
    finish(fig,f'05_{direction}_zone_maps')


In [ ]:
latest=int(z.year.max())
label_col=next((c for c in ['zone','Zone','NAME'] if c in zones.columns),None)
lookup=zones[['LocationID',label_col]] if label_col else zones[['LocationID']]
ranked=z[z.year==latest].merge(lookup,left_on='zone_id',right_on='LocationID',how='left')
fig,axes=plt.subplots(1,2,figsize=(16,7))
for ax,direction in zip(axes,DIRECTIONS):
    top=ranked[ranked.direction==direction].nlargest(15,'trips_per_day').sort_values('trips_per_day')
    labels=top[label_col].fillna(top.zone_id.astype(str)) if label_col else top.zone_id.astype(str)
    ax.barh(labels,top.trips_per_day)
    ax.set(title=f'{latest} {direction}: top 15 zones',xlabel='Trips per observed calendar day')
finish(fig,'05_top_zones')
fig,ax=plt.subplots()
for (direction,year),g in z.groupby(['direction','year']):
    shares=g.share.sort_values(ascending=False).to_numpy()
    ax.plot(np.arange(1,len(shares)+1),np.cumsum(shares)*100,label=f'{direction} {year}')
ax.set(title='Spatial concentration of activity',xlabel='Zones ranked by demand',ylabel='Cumulative share of trips (%)')
ax.legend(ncol=2,fontsize=8)
finish(fig,'05_demand_concentration')


In [ ]:
first,last=int(z.year.min()),int(z.year.max())
for direction in DIRECTIONS:
    d=z[z.direction==direction].pivot(index='zone_id',columns='year',values='share')
    change=((d[last]-d[first])*100).rename('share_change_pp').reset_index()
    layer=zones.merge(change,left_on='LocationID',right_on='zone_id',how='left')
    bound=max(abs(change.share_change_pp.min()),abs(change.share_change_pp.max()),.01)
    fig,ax=plt.subplots(figsize=(9,9))
    layer.plot(column='share_change_pp',ax=ax,cmap='BrBG',vmin=-bound,vmax=bound,legend=True,
               missing_kwds={'color':'lightgrey'},edgecolor='#808080',linewidth=.2)
    ax.set_title(f'{direction.title()} demand share: {last} minus {first} (percentage points)')
    ax.set_axis_off();finish(fig,f'05_{direction}_share_change')


## Findings to record
Distinguish absolute activity from demand shares. Review unmapped demand and common-month coverage. Destination maps describe trip destinations, not vehicle supply. Changes spanning 2023/2024 also cross a destination-date definition change; flag this explicitly.